# Exploring NHL API to get Data for Expected Goals Model

In [26]:
import pandas as pd
import requests
import json
from typing import List
import time
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

## Display

In [2]:
def adjust_df_display(dimension, action):
    """This function when called adjusts the output display of pandas dataframes. It either changes the max_columns or max_rows to infinite or resets those
    values to their default display limits.

    Args:
        dimension (string): Display dimension of a dataframe to alter; should be either "columns" or "rows".
        action (string): Action to be carried out on display settings; should be either "max" or "limit".
    """
    if dimension == "columns" and action == "max":
        pd.set_option('display.max_columns', None)
    elif dimension == "rows" and action == "max":
        pd.set_option('display.max_rows', None)
    elif dimension == "columns" and action == "limit":
        pd.reset_option('max_columns')
    else:
        pd.reset_option('max_rows')

In [3]:
adjust_df_display("columns", "max")
adjust_df_display("rows", "max")

## API Data

In [4]:
url = 'https://api-web.nhle.com/v1/gamecenter/2024020954/play-by-play'

# Send GET request to fetch game data
response = requests.get(url)

# Check if reponse was successful
if response.status_code == 200:
    # Parse JSON response
    data = response.json()
    # Pretty print the JSON with an indentation of 4 spaces
    pretty_json = json.dumps(data, indent=4)
    # Output data
    #print(pretty_json)
    
    # Step 2: Extract the plays list
    plays = data.get("plays", [])

    # Step 3: Flatten the data
    # We'll normalize both top-level and nested dictionaries
    #df = pd.json_normalize(plays, sep="_")
    df = pd.json_normalize(data, sep="_")
    
else:
    print(f"Error: {response.status_code}")

In [5]:
print(data.keys())

dict_keys(['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue', 'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset', 'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor', 'awayTeam', 'homeTeam', 'shootoutInUse', 'otInUse', 'clock', 'displayPeriod', 'maxPeriods', 'gameOutcome', 'plays', 'rosterSpots', 'regPeriods', 'summary'])


In [6]:
play_data = data.get("plays", [])
print(f"Total events: {len(play_data)}")

Total events: 336


In [7]:
# Inspect the first few events
for play in play_data[:3]:
    print(json.dumps(play, indent=2))

{
  "eventId": 52,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:00",
  "timeRemaining": "20:00",
  "situationCode": "1551",
  "homeTeamDefendingSide": "left",
  "typeCode": 520,
  "typeDescKey": "period-start",
  "sortOrder": 8
}
{
  "eventId": 51,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:00",
  "timeRemaining": "20:00",
  "situationCode": "1551",
  "homeTeamDefendingSide": "left",
  "typeCode": 502,
  "typeDescKey": "faceoff",
  "sortOrder": 11,
  "details": {
    "eventOwnerTeamId": 28,
    "losingPlayerId": 8482116,
    "winningPlayerId": 8477505,
    "xCoord": 0,
    "yCoord": 0,
    "zoneCode": "N"
  }
}
{
  "eventId": 103,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:09",
  "timeRemaining": "19:51",
  "situationCode": "1551",
  "homeTeamDefending

In [8]:
#print(json.dumps(data, indent=2))
#data

In [ ]:
print(json.dumps(plays, indent=2))  # Pretty-print entire event

In [8]:
df = pd.json_normalize(plays, sep='_')
df.head()

,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,periodDescriptor_number,periodDescriptor_periodType,periodDescriptor_maxRegulationPeriods,details_eventOwnerTeamId,details_losingPlayerId,details_winningPlayerId,details_xCoord,details_yCoord,details_zoneCode,details_shotType,details_shootingPlayerId,details_goalieInNetId,details_awaySOG,details_homeSOG,details_reason,details_hittingPlayerId,details_hitteePlayerId,details_playerId,details_blockingPlayerId,details_secondaryReason,details_typeCode,details_descKey,details_duration,details_committedByPlayerId,details_drawnByPlayerId,pptReplayUrl,details_scoringPlayerId,details_scoringPlayerTotal,details_assist1PlayerId,details_assist1PlayerTotal,details_awayScore,details_homeScore,details_highlightClipSharingUrl,details_highlightClipSharingUrlFr,details_highlightClip,details_highlightClipFr,details_discreteClip,details_discreteClipFr,details_assist2PlayerId,details_assist2PlayerTotal
0,52,00:00,20:00,1551,left,520,period-start,8,1,REG,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,51,00:00,20:00,1551,left,502,faceoff,11,1,REG,3,28.0,8482116.0,8477505.0,0.0,0.0,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,103,00:09,19:51,1551,left,506,shot-on-goal,12,1,REG,3,28.0,NaN,NaN,-55.0,1.0,O,wrist,8477505.0,8476999.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8,00:10,19:50,1551,left,516,stoppage,13,1,REG,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,goalie-stopped-after-sog,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,53,00:10,19:50,1551,left,502,faceoff,15,1,REG,3,9.0,8477505.0,8481596.0,-69.0,-22.0,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Event Type: 'goal'

In [9]:
df[df['typeDescKey'] == 'goal']

,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,periodDescriptor_number,periodDescriptor_periodType,periodDescriptor_maxRegulationPeriods,details_eventOwnerTeamId,details_losingPlayerId,details_winningPlayerId,details_xCoord,details_yCoord,details_zoneCode,details_shotType,details_shootingPlayerId,details_goalieInNetId,details_awaySOG,details_homeSOG,details_reason,details_hittingPlayerId,details_hitteePlayerId,details_playerId,details_blockingPlayerId,details_secondaryReason,details_typeCode,details_descKey,details_duration,details_committedByPlayerId,details_drawnByPlayerId,pptReplayUrl,details_scoringPlayerId,details_scoringPlayerTotal,details_assist1PlayerId,details_assist1PlayerTotal,details_awayScore,details_homeScore,details_highlightClipSharingUrl,details_highlightClipSharingUrlFr,details_highlightClip,details_highlightClipFr,details_discreteClip,details_discreteClipFr,details_assist2PlayerId,details_assist2PlayerTotal
57,264,11:05,08:55,1541,left,505,goal,171,1,REG,3,9.0,NaN,NaN,73.0,4.0,O,wrist,NaN,8477970.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8481596.0,12.0,8482092.0,10.0,0.0,1.0,https://nhl.com/video/sjs-ott-pinto-scores-shg...,https://nhl.com/fr/video/sjs-ott-pinto-marque-...,6.369509e+12,6.369509e+12,6.369509e+12,6.369507e+12,NaN,NaN
125,579,05:17,14:43,1541,right,505,goal,350,2,REG,3,28.0,NaN,NaN,83.0,23.0,O,wrist,NaN,8476999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8475726.0,22.0,8484227.0,18.0,1.0,1.0,https://nhl.com/video/sjs-ott-toffoli-scores-g...,NaN,6.369510e+12,NaN,NaN,NaN,8484801.0,25.0
157,641,09:47,10:13,1541,right,505,goal,406,2,REG,3,28.0,NaN,NaN,37.0,4.0,O,slap,NaN,8476999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8480043.0,5.0,8477505.0,18.0,2.0,1.0,https://nhl.com/video/sjs-ott-liljegren-scores...,https://nhl.com/fr/video/sjs-ott-liljegren-mar...,6.369510e+12,6.369509e+12,6.369508e+12,6.369510e+12,8480011.0,5.0
228,832,01:26,18:34,1351,left,505,goal,551,3,REG,3,9.0,NaN,NaN,56.0,-16.0,O,wrist,NaN,8477970.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8480801.0,22.0,8482116.0,42.0,2.0,2.0,https://nhl.com/video/sjs-ott-tkachuk-scores-p...,NaN,6.369509e+12,NaN,6.369510e+12,6.369509e+12,8482105.0,31.0
234,844,03:00,17:00,1551,left,505,goal,563,3,REG,3,9.0,NaN,NaN,77.0,-13.0,O,wrist,NaN,8477970.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8482116.0,19.0,8482092.0,11.0,2.0,3.0,https://nhl.com/video/sjs-ott-stutzle-scores-g...,https://nhl.com/fr/video/sjs-ott-stutzle-marqu...,6.369512e+12,6.369510e+12,6.369510e+12,6.369510e+12,NaN,NaN
278,937,08:31,11:29,1551,left,505,goal,661,3,REG,3,9.0,NaN,NaN,84.0,7.0,O,wrist,NaN,8477970.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8474102.0,2.0,8478469.0,24.0,2.0,4.0,https://nhl.com/video/sjs-ott-perron-scores-go...,https://nhl.com/fr/video/sjs-ott-perron-marque...,6.369512e+12,6.369510e+12,6.369513e+12,6.369512e+12,8480208.0,31.0
327,1071,18:33,01:27,0641,left,505,goal,802,3,REG,3,28.0,NaN,NaN,-80.0,-7.0,O,wrist,NaN,8476999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8484227.0,10.0,8482667.0,31.0,3.0,4.0,https://nhl.com/video/sjs-ott-smith-scores-ppg...,https://nhl.com/fr/video/will-smith-with-a-pow...,6.369511e+12,6.369511e+12,6.369512e+12,6.369511e+12,8484801.0,26.0
329,1078,19:00,01:00,0651,left,505,goal,808,3,REG,3,9.0,NaN,NaN,20.0,-29.0,N,wrist,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://wsr.nhle.com/sprites/20242025/20240209...,8478020.0,6.0,8482245.0,7.0,3.0,5.0,https://nhl.com/video/sjs-ott-amadio-scores-em...,https://nhl.com/fr/video/sjs-ott-amadio-ott-ma...,6.

In [10]:
df[df['typeDescKey'] == 'shot-on-goal'].head()

,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,periodDescriptor_number,periodDescriptor_periodType,periodDescriptor_maxRegulationPeriods,details_eventOwnerTeamId,details_losingPlayerId,details_winningPlayerId,details_xCoord,details_yCoord,details_zoneCode,details_shotType,details_shootingPlayerId,details_goalieInNetId,details_awaySOG,details_homeSOG,details_reason,details_hittingPlayerId,details_hitteePlayerId,details_playerId,details_blockingPlayerId,details_secondaryReason,details_typeCode,details_descKey,details_duration,details_committedByPlayerId,details_drawnByPlayerId,pptReplayUrl,details_scoringPlayerId,details_scoringPlayerTotal,details_assist1PlayerId,details_assist1PlayerTotal,details_awayScore,details_homeScore,details_highlightClipSharingUrl,details_highlightClipSharingUrlFr,details_highlightClip,details_highlightClipFr,details_discreteClip,details_discreteClipFr,details_assist2PlayerId,details_assist2PlayerTotal
2,103,00:09,19:51,1551,left,506,shot-on-goal,12,1,REG,3,28.0,NaN,NaN,-55.0,1.0,O,wrist,8477505.0,8476999.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,131,02:11,17:49,1551,left,506,shot-on-goal,40,1,REG,3,28.0,NaN,NaN,-56.0,11.0,O,wrist,8480848.0,8476999.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,140,02:57,17:03,1551,left,506,shot-on-goal,52,1,REG,3,28.0,NaN,NaN,-62.0,-15.0,O,wrist,8484911.0,8476999.0,3.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,148,03:34,16:26,1551,left,506,shot-on-goal,61,1,REG,3,9.0,NaN,NaN,51.0,-18.0,O,wrist,8480801.0,8477970.0,3.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,155,03:59,16:01,1551,left,506,shot-on-goal,67,1,REG,3,28.0,NaN,NaN,-54.0,32.0,O,wrist,8482144.0,8476999.0,4.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df[df['typeDescKey'] == 'missed-shot'].head()

,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,periodDescriptor_number,periodDescriptor_periodType,periodDescriptor_maxRegulationPeriods,details_eventOwnerTeamId,details_losingPlayerId,details_winningPlayerId,details_xCoord,details_yCoord,details_zoneCode,details_shotType,details_shootingPlayerId,details_goalieInNetId,details_awaySOG,details_homeSOG,details_reason,details_hittingPlayerId,details_hitteePlayerId,details_playerId,details_blockingPlayerId,details_secondaryReason,details_typeCode,details_descKey,details_duration,details_committedByPlayerId,details_drawnByPlayerId,pptReplayUrl,details_scoringPlayerId,details_scoringPlayerTotal,details_assist1PlayerId,details_assist1PlayerTotal,details_awayScore,details_homeScore,details_highlightClipSharingUrl,details_highlightClipSharingUrlFr,details_highlightClip,details_highlightClipFr,details_discreteClip,details_discreteClipFr,details_assist2PlayerId,details_assist2PlayerTotal
6,115,01:09,18:51,1551,left,507,missed-shot,26,1,REG,3,9.0,NaN,NaN,41.0,37.0,O,wrist,8480801.0,8477970.0,NaN,NaN,high-and-wide-left,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,116,01:12,18:48,1551,left,507,missed-shot,28,1,REG,3,9.0,NaN,NaN,54.0,-39.0,O,wrist,8482245.0,8477970.0,NaN,NaN,wide-left,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,186,06:06,13:54,1551,left,507,missed-shot,98,1,REG,3,28.0,NaN,NaN,-50.0,-2.0,O,wrist,8477505.0,8476999.0,NaN,NaN,wide-right,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,57,07:26,12:34,1551,left,507,missed-shot,113,1,REG,3,28.0,NaN,NaN,-71.0,-11.0,O,tip-in,8475726.0,8476999.0,NaN,NaN,wide-right,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,208,07:41,12:19,1551,left,507,missed-shot,122,1,REG,3,28.0,NaN,NaN,-85.0,8.0,O,backhand,8479316.0,8476999.0,NaN,NaN,wide-left,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Old version
def get_new_shot_events(game_id: str) -> List[dict]:
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch {game_id}")
        return []

    data = response.json()
    plays = data.get('plays', [])
    
    shot_events = []
    for play in plays:
        event_type = play.get('typeDescKey')
        event_id = play.get('eventId')
        if event_type in ['shot-on-goal', 'missed-shot', 'goal']:
            coordinates = play.get('coordinates', {})
            players = play.get('details', {}).get('players', [])
            
            shooter = next((p['player']['fullName'] for p in players if p['playerType'] in ['Shooter', 'Scorer']), None)
            goalie = next((p['player']['fullName'] for p in players if p['playerType'] == 'Goalie'), None)

            shot_events.append({
                'game_id': game_id,
                'eventId': event_id,
                'event_type': event_type,
                'period': play.get('period'),
                'period_time': play.get('timeInPeriod'),
                'team': play.get('team', {}).get('name'),
                'x': coordinates.get('x'),
                'y': coordinates.get('y'),
                'shooter': shooter,
                'goalie': goalie
            })
    
    return shot_events

In [15]:
# Example usage
game_ids = ["2024020954"]
all_shots = []
for gid in game_ids:
    all_shots.extend(get_new_shot_events(gid))

shots_df = pd.DataFrame(all_shots)
shots_df
#print(shots_df.head())

,game_id,eventId,event_type,period,period_time,team,x,y,shooter,goalie
0,2024020954,103,shot-on-goal,None,00:09,None,None,None,None,None
1,2024020954,115,missed-shot,None,01:09,None,None,None,None,None
2,2024020954,116,missed-shot,None,01:12,None,None,None,None,None
3,2024020954,131,shot-on-goal,None,02:11,None,None,None,None,None
4,2024020954,140,shot-on-goal,None,02:57,None,None,None,None,None
5,2024020954,148,shot-on-goal,None,03:34,None,None,None,None,None
6,2024020954,155,shot-on-goal,None,03:59,None,None,None,None,None
7,2024020954,162,shot-on-goal,None,04:29,None,None,None,None,None
8,2024020954,186,missed-shot,None,06:06,None,None,None,None,None
9,2024020954,199,shot-on-goal,None,07:16,None,None,None,None,None


In [16]:
game_id = "2024020954"
url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
response = requests.get(url)
data = response.json()

# # Filter for a single shot event
# for play in data.get("plays", []):
#     if play.get("typeDescKey") in ["shot-on-goal", "goal", "missed-shot", "blocked-shot"]:
#         print(json.dumps(play, indent=2))  # Pretty print the full structure
#         break  # Stop after first match
count = 0
for play in data.get("plays", []):
    print(play.keys())
    if play.get("typeDescKey") in ["shot-on-goal", "goal", "missed-shot"]:
        print(json.dumps(play, indent=2))  # Pretty-print entire event
        print("=" * 80)  # Visual separator
        count += 1
        if count == 1:
            break

dict_keys(['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder'])
dict_keys(['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder', 'details'])
dict_keys(['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder', 'details'])
{
  "eventId": 103,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "00:09",
  "timeRemaining": "19:51",
  "situationCode": "1551",
  "homeTeamDefendingSide": "left",
  "typeCode": 506,
  "typeDescKey": "shot-on-goal",
  "sortOrder": 12,
  "details": {
    "xCoord": -55,
    "yCoord": 1,
    "zoneCode": "O",
    "shotType": "wrist",
    "shootingPlayerId": 8477505,
    "goalieInNetId": 8476999,
    "eventOwnerTeamId": 28,
    "aw

In [ ]:
# Old version
def extract_shot_events(play_data: list) -> list[dict]:
    """
    Extracts relevant information from shot-related events (goal, shot-on-goal, missed-shot).
    Returns a list of dictionaries with a consistent schema.
    """
    shot_events = []
    valid_types = {"goal", "shot-on-goal", "missed-shot"}

    for event in play_data:
        event_type = event.get("typeDescKey")
        if event_type not in valid_types:
            continue
        
        details = event.get("details", {})
        shooter_id = (
            details.get("scoringPlayerId") if event_type == "goal"
            else details.get("shootingPlayerId")
        )
        
        try:
            shot_info = {
                "event_type": event_type,
                "period": event.get("periodDescriptor", {}).get("number"),
                "time": event.get("timeInPeriod"),
                "x_coord": details.get("xCoord"),
                "y_coord": details.get("yCoord"),
                "zone": details.get("zoneCode"),
                "shot_type": details.get("shotType"),
                "shooter_id": shooter_id,
                "goalie_id": details.get("goalieInNetId"),
                "team_id": details.get("eventOwnerTeamId"),
                "is_goal": 1 if event_type == "goal" else 0
            }
            shot_events.append(shot_info)
        except Exception as e:
            print(f"Skipping event due to error: {e}")
            continue

    return shot_events

### Game Data

In [17]:
def get_nhl_team_abbreviations() -> list[str]:
    url = "https://api-web.nhle.com/v1/standings/now"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    abbrevs = []
    for team_record in data.get("standings", []):
        team_abbrev_info = team_record.get("teamAbbrev", {})
        abbrev = team_abbrev_info.get("default")
        if abbrev:
            abbrevs.append(abbrev)

    return sorted(set(abbrevs))

# Example usage
team_abbrevs = get_nhl_team_abbreviations()

In [18]:
team_abbrevs

['ANA',
 'BOS',
 'BUF',
 'CAR',
 'CBJ',
 'CGY',
 'CHI',
 'COL',
 'DAL',
 'DET',
 'EDM',
 'FLA',
 'LAK',
 'MIN',
 'MTL',
 'NJD',
 'NSH',
 'NYI',
 'NYR',
 'OTT',
 'PHI',
 'PIT',
 'SEA',
 'SJS',
 'STL',
 'TBL',
 'TOR',
 'UTA',
 'VAN',
 'VGK',
 'WPG',
 'WSH']

In [ ]:
def get_all_regular_season_game_ids(season: str, team_abbrevs: list[str]):
    game_ids = set()
    for team in team_abbrevs:
        url = f"https://api-web.nhle.com/v1/club-schedule-season/{team}/{season}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        
        for g in data.get("games", []):
            if g.get("gameType") == 2:
                game_ids.add(g["id"])
    return sorted(game_ids)

# Example usage
season = '20232024'
game_ids = get_all_regular_season_game_ids(season, team_abbrevs)
print(f"Found {len(game_ids)} regular-season games for {season}")

Found 1312 regular-season games for 20232024


In [21]:
url = f"https://api-web.nhle.com/v1/club-schedule-season/{'OTT'}/{'20232024'}"
response = requests.get(url)
response.raise_for_status()
data = response.json()

In [34]:
import pandas as pd
import requests

url = f"https://api-web.nhle.com/v1/club-schedule-season/OTT/20232024"
response = requests.get(url)
response.raise_for_status()
data = response.json()

games = data['games']  # assuming the API wraps this list

created_at_utc = datetime.now(timezone.utc)
created_at_et = created_at_utc.astimezone(ZoneInfo("America/Toronto"))

rows = []

for game in games:
    row = {
        "game_id": game["id"],
        "season": game["season"],
        "game_type": game["gameType"],
        "game_date": game["gameDate"],
        "game_state": game["gameState"],
        "game_schedule_state": game["gameScheduleState"],
        
        # Away team
        "away_team_id": game["awayTeam"]["id"],
        "away_team_city": game["awayTeam"]["placeName"]["default"],
        "away_team_name": game["awayTeam"]["commonName"]["default"],
        "away_team_abbrev": game["awayTeam"]["abbrev"],
        "away_team_score": game["awayTeam"]["score"],
        
        # Home team
        "home_team_id": game["homeTeam"]["id"],
        "home_team_city": game["homeTeam"]["placeName"]["default"],
        "home_team_name": game["homeTeam"]["commonName"]["default"],
        "home_team_abbrev": game["homeTeam"]["abbrev"],
        "home_team_score": game["homeTeam"]["score"],
        
        # Outcome
        "last_period_type": game.get("gameOutcome", {}).get("lastPeriodType"),
        
        # Created at
        "created_at_utc": created_at_utc,
        "created_at_et": created_at_et
    }
    rows.append(row)

df = pd.DataFrame(rows)
df.columns

Index(['game_id', 'season', 'game_type', 'game_date', 'game_state',
       'game_schedule_state', 'away_team_id', 'away_team_city',
       'away_team_name', 'away_team_abbrev', 'away_team_score', 'home_team_id',
       'home_team_city', 'home_team_name', 'home_team_abbrev',
       'home_team_score', 'last_period_type', 'created_at_utc',
       'created_at_et'],
      dtype='object')

In [36]:
df.head()

,game_id,season,game_type,game_date,game_state,game_schedule_state,away_team_id,away_team_city,away_team_name,away_team_abbrev,away_team_score,home_team_id,home_team_city,home_team_name,home_team_abbrev,home_team_score,last_period_type,created_at_utc,created_at_et
0,2023010007,20232024,1,2023-09-24,FINAL,OK,10,Toronto,Maple Leafs,TOR,2,9,Ottawa,Senators,OTT,3,REG,2026-03-26 01:27:37.062335+00:00,2026-03-25 21:27:37.062335-04:00
1,2023010019,20232024,1,2023-09-25,FINAL,OK,9,Ottawa,Senators,OTT,4,10,Toronto,Maple Leafs,TOR,3,OT,2026-03-26 01:27:37.062335+00:00,2026-03-25 21:27:37.062335-04:00
2,2023010034,20232024,1,2023-09-27,FINAL,OK,9,Ottawa,Senators,OTT,3,8,Montréal,Canadiens,MTL,4,REG,2026-03-26 01:27:37.062335+00:00,2026-03-25 21:27:37.062335-04:00
3,2023010052,20232024,1,2023-09-29,FINAL,OK,52,Winnipeg,Jets,WPG,1,9,Ottawa,Senators,OTT,3,REG,2026-03-26 01:27:37.062335+00:00,2026-03-25 21:27:37.062335-04:00
4,2023010065,20232024,1,2023-10-01,FINAL,OK,13,Florida,Panthers,FLA,2,9,Ottawa,Senators,OTT,4,REG,2026-03-26 01:27:37.062335+00:00,2026-03-25 21:27:37.062335-04:00


In [41]:
df['game_state'].unique()

array(['FINAL', 'OFF'], dtype=object)

In [37]:
df.dtypes

game_id                                          int64
season                                           int64
game_type                                        int64
game_date                                       object
game_state                                      object
game_schedule_state                             object
away_team_id                                     int64
away_team_city                                  object
away_team_name                                  object
away_team_abbrev                                object
away_team_score                                  int64
home_team_id                                     int64
home_team_city                                  object
home_team_name                                  object
home_team_abbrev                                object
home_team_score                                  int64
last_period_type                                object
created_at_utc                     datetime64[ns, UTC]
created_at

### Shot Data